In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm


In [ ]:
import pandas as pd
from pathlib import Path

def load_complete_dataset(repo_root, sample_size=None):
    """
    Load and merge the complete YouTube dataset efficiently using chunked processing.
    
    Args:
        repo_root: Path to repository root
        sample_size: Optional limit for testing (None = full dataset)
    
    Returns:
        df_merged: Complete merged DataFrame ready for analysis
    """
    print("=== LOADING COMPLETE DATASET ===")
    
    # Step 1: Build video metadata lookup dict
    print("Step 1: Loading video metadata...")
    video_to_meta = {}
    meta_chunks = pd.read_json(
        repo_root/"dataset"/"zip"/"yt_metadata_en.jsonl.gz",
        lines=True,
        compression="gzip", 
        chunksize=1_000_000  # Process 1M records at a time
    )
    
    processed_meta = 0
    meta_progress = tqdm(desc="Processing metadata chunks", unit="chunk")
    
    for chunk in meta_chunks:
        chunk_progress = tqdm(chunk.iterrows(), total=len(chunk), desc=f"Chunk {meta_progress.n+1}", leave=False)
        
        for _, row in chunk_progress:
            # Only store videos with valid categories and dates
            if pd.notna(row['categories']) and row['categories'] != '':
                video_to_meta[row['display_id']] = {
                    'title': row['title'],
                    'categories': row['categories'],
                    'upload_date': row['upload_date'],
                    'view_count': row.get('view_count', 0),
                    'duration': row.get('duration', 0)
                }
        
        processed_meta += len(chunk)
        meta_progress.update(1)
        meta_progress.set_postfix({"Records": f"{processed_meta:,}", "Stored": f"{len(video_to_meta):,}"})
        
        if sample_size and processed_meta >= sample_size:
            break
    
    meta_progress.close()
    print(f"✓ Metadata loaded: {len(video_to_meta):,} valid videos")
    
    # Step 2: Load URLs and merge with metadata
    print("\nStep 2: Loading URLs and merging...")
    merged_records = []
    batch_buffer = []
    batch_size = 50_000  # Write in batches to manage memory
    
    url_chunks = pd.read_json(
        repo_root/"dataset"/"urls.jsonl.gz",
        lines=True,
        compression="gzip", 
        chunksize=1_000_000
    )
    
    processed_urls = 0
    matched_count = 0
    url_progress = tqdm(desc="Processing URL chunks", unit="chunk")
    
    for chunk in url_chunks:
        chunk_progress = tqdm(chunk.iterrows(), total=len(chunk), desc=f"Chunk {url_progress.n+1}", leave=False)
        
        for _, row in chunk_progress:
            video_id = row['display_id']
            
            # Check if we have metadata for this video
            if video_id in video_to_meta:
                meta = video_to_meta[video_id]
                
                # Create merged record
                merged_record = {
                    'display_id': video_id,
                    'urls': row['urls'],
                    'title': meta['title'],
                    'categories': meta['categories'],
                    'upload_date': meta['upload_date'],
                    'view_count': meta['view_count'],
                    'duration': meta['duration']
                }
                
                batch_buffer.append(merged_record)
                matched_count += 1
                
                # Process batch when buffer gets full
                if len(batch_buffer) >= batch_size:
                    merged_records.extend(batch_buffer)
                    batch_buffer = []
        
        processed_urls += len(chunk)
        url_progress.update(1)
        url_progress.set_postfix({"Processed": f"{processed_urls:,}", "Matched": f"{matched_count:,}"})
        
        if sample_size and processed_urls >= sample_size:
            break
    
    # Add remaining buffer
    merged_records.extend(batch_buffer)
    url_progress.close()
    
    print(f"✓ URLs processed: {processed_urls:,} total, {matched_count:,} matched ({matched_count/processed_urls*100:.1f}%)")
    
    # Step 3: Convert to DataFrame and clean
    print("\nStep 3: Creating final DataFrame...")
    df_merged = pd.DataFrame(merged_records)
    
    with tqdm(total=4, desc="Cleaning data") as pbar:
        # Clean and process dates
        df_merged['upload_date'] = pd.to_datetime(df_merged['upload_date'], errors='coerce')
        pbar.update(1)
        
        df_merged['year'] = df_merged['upload_date'].dt.year
        pbar.update(1)
        
        # Remove rows with invalid dates
        df_merged = df_merged.dropna(subset=['upload_date']).copy()
        pbar.update(1)
        
        # Clear the lookup dict to free memory
        del video_to_meta
        pbar.update(1)
    
    print(f"✓ Final dataset: {len(df_merged):,} records")
    if len(df_merged) > 0:
        print(f"  Years: {df_merged['year'].min():.0f} - {df_merged['year'].max():.0f}")
        print(f"  Categories: {df_merged['categories'].nunique()} unique")
        print(f"  Memory usage: {df_merged.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    
    return df_merged

# Usage examples:
# For testing (sample):
# df_merged = load_complete_dataset(repo_root, sample_size=1_000_000)

# For full dataset:
# df_merged = load_complete_dataset(repo_root)

In [13]:
# Debug: Check if files exist and examine structure
repo_root = Path.cwd().parent.parent

print(repo_root)
path_urls = repo_root/"dataset"/"urls.jsonl.gz"
path_meta = repo_root/"dataset"/"zip"/"yt_metadata_en.jsonl.gz"

#df_merged = load_complete_dataset(path_urls, sample_size=1_000_000)

print("Checking files...")
metadata_path = repo_root / "dataset" / "zip" / "yt_metadata_en.jsonl.gz"
urls_path = repo_root / "dataset" / "urls.jsonl.gz"

print(f"Metadata file exists: {metadata_path.exists()}")
print(f"URLs file exists: {urls_path.exists()}")

if metadata_path.exists():
    print(f"Metadata file size: {metadata_path.stat().st_size / 1024**2:.1f} MB")

if urls_path.exists():
    print(f"URLs file size: {urls_path.stat().st_size / 1024**2:.1f} MB")

c:\Users\veron\Desktop\ma1\ada-2025-project-shenanigans
Checking files...
Metadata file exists: True
URLs file exists: True
Metadata file size: 13004.4 MB
URLs file size: 1054.1 MB


In [14]:
df_merged = load_complete_dataset(repo_root, sample_size=1_000_000)


=== LOADING COMPLETE DATASET ===
Step 1: Loading video metadata...
  Processed 1,000,000 metadata records, stored 999,993
✓ Metadata loaded: 999,993 valid videos

Step 2: Loading URLs and merging...


KeyboardInterrupt: 